# Executive Summary

This sales analysis project examines over **186,000** customer orders to identify revenue trends, product performance, customer purchasing behavior, and operational opportunities.

        Key Business Findings
- Total sales revenue reached **$34.49M** in 2019.
- **December** generated the highest revenue, while **January** had the lowest sales.
- The top 5 products contributed nearly **66% of total revenue**, creating revenue concentration risk.
- Peak customer purchasing activity occurred between **11 AM and 8 PM**.
- **San Francisco** generated the highest revenue among all cities.
- Customers frequently purchased **phones together with charging accessories**, revealing strong bundling opportunities.

In [1]:
import os
import calendar
import warnings
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [2]:
files = [file for file in os.listdir() if file.endswith('.csv')]
df = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)

## Understanding & Data Preperation

### 1️⃣ Data Cleaning

     Finding Empty Values

In [3]:
df.isna().sum()

Order ID            545
Product             545
Quantity Ordered    545
Price Each          545
Order Date          545
Purchase Address    545
dtype: int64

     Delete Empty Values

In [4]:
df = df.dropna(how='all')

### 2️⃣ Gathering Information

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 186305 entries, 0 to 186849
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Order ID          186305 non-null  object
 1   Product           186305 non-null  object
 2   Quantity Ordered  186305 non-null  object
 3   Price Each        186305 non-null  object
 4   Order Date        186305 non-null  object
 5   Purchase Address  186305 non-null  object
dtypes: object(6)
memory usage: 9.9+ MB


In [6]:
df.describe(include='all')

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
count,186305,186305,186305,186305,186305,186305
unique,178438,20,10,24,142396,140788
top,Order ID,USB-C Charging Cable,1,11.95,Order Date,Purchase Address
freq,355,21903,168552,21903,355,355


     Deleting String Values From Data

In [7]:
df = df[df['Order ID'] != 'Order ID']
df.reset_index(drop=True,inplace=True)

### 3️⃣ Conversion

    Convert String Values to Numerical

In [8]:
df['Order ID'] = df['Order ID'].astype(int)
df['Quantity Ordered'] = df['Quantity Ordered'].astype(int)
df['Price Each'] = df['Price Each'].astype(float)

    Convert String to DateTime

In [9]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format = "%m/%d/%y %H:%M")

### 4️⃣ Add Columns

    Add Total Revenue Column

In [10]:
df['Total Revenue'] = df['Quantity Ordered'] * df['Price Each']

    Add Date, Month, Year, Hour, Minutes Columns

In [11]:
df['Date'] = df['Order Date'].dt.date
df['Month'] = df['Order Date'].dt.month
df['Year'] = df['Order Date'].dt.year
df['Hour'] = df['Order Date'].dt.hour
df['Minute'] = df['Order Date'].dt.minute

    Add City Column

In [12]:
df['City'] = df['Purchase Address'].apply(lambda x:x.split(',')[1] + f" ({x.split(',')[2].split(" ")[1]})")

## Data Analysis

### 1️⃣ Revenue Performance

    What is the total revenue for 2019?

In [13]:
total_revenue = df['Total Revenue'].sum()
print(f"Total Revenue of 2019 are: ${total_revenue:,.0f}")

Total Revenue of 2019 are: $34,492,036


    What is the monthly revenue trend?

In [14]:
#   Monthly Revenue
monthly_revenue_trend = df.groupby('Month')['Total Revenue'].sum().reset_index()

#   Add Months name instead of Numbers
monthly_revenue_trend['Month'] = monthly_revenue_trend['Month'].apply(lambda x: calendar.month_abbr[x])

In [15]:
monthly_revenue = monthly_revenue_trend.copy()

#   Write in Proper Format (in comma's form)
monthly_revenue['Total Revenue'] = monthly_revenue['Total Revenue'].apply(lambda x: f"${x:,.0f}")

#   Arrange in Order (Highest to Lowest)
monthly_revenue.sort_values(by='Total Revenue', ascending=False)

,Month,Total Revenue
11,Dec,"$4,613,443"
9,Oct,"$3,736,727"
3,Apr,"$3,390,670"
10,Nov,"$3,199,603"
4,May,"$3,152,607"
2,Mar,"$2,807,100"
6,Jul,"$2,647,776"
5,Jun,"$2,577,802"
7,Aug,"$2,244,468"
1,Feb,"$2,202,022"


In [16]:
fig = go.Figure()

text_positions = ['top center'] * len(monthly_revenue_trend)
text_positions[0] = 'top right'
text_positions[-1] = 'top left'

fig.add_trace(go.Scatter(
    x=monthly_revenue_trend['Month'],
    y=monthly_revenue_trend['Total Revenue'],
    
    mode='lines+markers+text',
    line=dict(
        color='#1E88E5',
        width=4,
        shape='spline'),
    marker=dict(
        size=8,
        color='#0A2540',
        line=dict(width=2, color='white')),

    text=monthly_revenue_trend['Total Revenue'].apply(lambda x: f"<b>{x/1e6:.1f}M</b>"),
    textposition=text_positions,
    fill='tozeroy',
    fillcolor='rgba(30, 136, 229, 0.15)',

    hovertemplate='<b>Month:</b> %{x}<br><b>Revenue:</b> $%{y:,.0f}<extra></extra>'
))
fig.update_layout(
    title=dict(text="<b>Sales Revenue Trend by Month</b>", x=0.5, font=dict(size=24)),

    xaxis=dict(
        title=dict(text='<b>Months</b>', font=dict(size=18))),

    yaxis=dict(
        title=dict(text='<b>Total Revenue</b>', font=dict(size=18)),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.15)'),
        
    template='simple_white',
    hovermode='x unified')
fig.show()

    Calculate Month-over-Month growth %

In [17]:
# Month over Month Revenue
month_o_month = df.groupby('Month')['Total Revenue'].sum().reset_index()

# Add Month names
month_o_month['Month'] = month_o_month['Month'].apply(lambda x: calendar.month_abbr[x])

# Add MOM Growth % Column
month_o_month['MOM Growth %'] = month_o_month['Total Revenue'].pct_change() * 100

# Drop January (NaN growth)
month_o_month.drop(index=0, inplace=True)

m_o_m = month_o_month.copy()

# Write in Proper Readable Format
month_o_month['Total Revenue'] = month_o_month['Total Revenue'].apply(lambda x: f"${x:,.2f}")

# Write in Proper Readable Format
month_o_month['MOM Growth %'] = month_o_month['MOM Growth %'].apply(lambda x: f"{x:.2f}%")

month_o_month

,Month,Total Revenue,MOM Growth %
1,Feb,"$2,202,022.42",20.84%
2,Mar,"$2,807,100.38",27.48%
3,Apr,"$3,390,670.24",20.79%
4,May,"$3,152,606.75",-7.02%
5,Jun,"$2,577,802.26",-18.23%
6,Jul,"$2,647,775.76",2.71%
7,Aug,"$2,244,467.88",-15.23%
8,Sep,"$2,097,560.13",-6.55%
9,Oct,"$3,736,726.88",78.15%
10,Nov,"$3,199,603.20",-14.37%


    Best month, Worst month

In [18]:
best_month = m_o_m.loc[m_o_m['MOM Growth %'].idxmax()]
worst_month = m_o_m.loc[m_o_m['MOM Growth %'].idxmin()]

print(f"Best Month: {best_month['Month']} ({best_month['MOM Growth %']:.2f}%)")
print(f"Worst Month: {worst_month['Month']} ({worst_month['MOM Growth %']:.2f}%)")

Best Month: Oct (78.15%)
Worst Month: Jun (-18.23%)


In [19]:
month_o_month = df.groupby('Month')['Total Revenue'].sum().reset_index()
month_o_month['Month'] = month_o_month['Month'].apply(lambda x: calendar.month_abbr[x])
month_o_month['MOM Growth %'] = month_o_month['Total Revenue'].pct_change() * 100
month_o_month.drop(index=0,inplace=True)


colors = ['green' if x > 0 else 'red' for x in month_o_month['MOM Growth %']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x = month_o_month['Month'],
    y = month_o_month['MOM Growth %'],

    marker_color = colors,
    text=month_o_month['MOM Growth %'].apply(lambda x: f"<b>{x:.1f}%</b>"),
    textposition='outside',

    hovertemplate='<b>Month:</b> %{x}<br><b>Growth:</b> %{y:,.1f}%<extra></extra>'
    ))
fig.update_layout(
    title=dict(text='<b>📈 Month-over-Month Growth %</b>', x=0.5, font=dict(size=18)),
    xaxis=dict(title=dict(text='<b>Months</b>', font=dict(size=16))),
    yaxis=dict(title=dict(text='<b>📈 Growth (%)</b>', font=dict(size=16)), showgrid=True, gridcolor='rgba(0,0,0,0.15)',
    range=[-25, 80]
    ),
    template='simple_white',
)
fig.show()

### 2️⃣ Product Performance

    Revenue by Product

In [20]:
# Each Product Revenue
product_revenue = df.groupby('Product')['Total Revenue'].sum().reset_index()

# Arrange in Order (Highest to Lowest)
product_revenue = product_revenue.sort_values(by='Total Revenue',ascending=False)

# Write in Proper Readable Format
product_revenue['Total Revenue'] = product_revenue['Total Revenue'].apply(lambda x: f'${x:,.0f}')

product_revenue

,Product,Total Revenue
13,Macbook Pro Laptop,"$8,037,600"
18,iPhone,"$4,794,300"
14,ThinkPad Laptop,"$4,129,959"
9,Google Phone,"$3,319,200"
1,27in 4K Gaming Monitor,"$2,435,098"
3,34in Ultrawide Monitor,"$2,355,558"
6,Apple Airpods Headphones,"$2,349,150"
8,Flatscreen TV,"$1,445,700"
7,Bose SoundSport Headphones,"$1,345,565"
2,27in FHD Monitor,"$1,132,424"


    Quantity sold by product

In [47]:
# Number of Product Sold
quantity_sold_product = df.groupby('Product')['Quantity Ordered'].sum().reset_index()

# Arrange in Order (Highest to Lowest)
quantity_sold_product = quantity_sold_product.sort_values(by='Quantity Ordered',ascending=False)

# Write in Proper Readable Format
quantity_sold_product['Quantity Ordered'] = quantity_sold_product['Quantity Ordered'].apply(lambda x: f'{x:,.0f}')

quantity_sold_product

,Product,Quantity Ordered
5,AAA Batteries (4-pack),"30,372"
4,AA Batteries (4-pack),"27,136"
15,USB-C Charging Cable,"22,109"
12,Lightning Charging Cable,"21,715"
17,Wired Headphones,"19,120"
6,Apple Airpods Headphones,"15,016"
7,Bose SoundSport Headphones,"12,894"
2,27in FHD Monitor,"7,393"
18,iPhone,"6,729"
1,27in 4K Gaming Monitor,"6,118"


    Top 5 products by revenue

In [22]:
top_5_product_revenue = df.groupby('Product')['Total Revenue'].sum().reset_index()

#Arrange in Order (Highest to Lowest)
top_5_product_revenue = top_5_product_revenue.sort_values(by='Total Revenue',ascending=False).head(5)

# Write in Proper Readable Format
top_5_product_revenue['Total Revenue'] = top_5_product_revenue['Total Revenue'].apply(lambda x: f'${x:,.0f}')

top_5_product_revenue

,Product,Total Revenue
13,Macbook Pro Laptop,"$8,037,600"
18,iPhone,"$4,794,300"
14,ThinkPad Laptop,"$4,129,959"
9,Google Phone,"$3,319,200"
1,27in 4K Gaming Monitor,"$2,435,098"


    Bottom 5 products by revenue

In [23]:
bottom_5_product_revenue = df.groupby('Product')['Total Revenue'].sum().reset_index()

# Arrange in Order (Highest to Lowest)
bottom_5_product_revenue = bottom_5_product_revenue.sort_values(by='Total Revenue',ascending=True).head(5)

# Write in Proper Readable Format
bottom_5_product_revenue['Total Revenue'] = bottom_5_product_revenue['Total Revenue'].apply(lambda x: f'${x:,.0f}')

bottom_5_product_revenue

,Product,Total Revenue
5,AAA Batteries (4-pack),"$92,741"
4,AA Batteries (4-pack),"$106,118"
17,Wired Headphones,"$246,478"
15,USB-C Charging Cable,"$286,501"
12,Lightning Charging Cable,"$347,094"


    Quantity Sold vs Product Revenue

In [24]:
quantity_sold_product = df.groupby('Product')['Quantity Ordered'].sum().reset_index()
product_revenue = df.groupby('Product')['Total Revenue'].sum().reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=quantity_sold_product['Product'],
    y=quantity_sold_product['Quantity Ordered'],

    text=quantity_sold_product['Quantity Ordered'],
    texttemplate='<b>%{text:,}</b>',
    textposition='outside',

    marker=dict(color=quantity_sold_product['Quantity Ordered']),
    yaxis='y1',
    name='<b>Quantity Ordered</b>',

    hovertemplate='<b>Quantity Ordered:</b> %{y:,.0f}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=product_revenue['Product'],
    y=product_revenue['Total Revenue'],

#
line=dict(
        color='green',
        width=2,
        shape='spline'),
    marker=dict(
        size=6,
        color='#0A2540',
        line=dict(width=1, color='white')),


    fill='tozeroy',
    fillcolor='rgba(60, 234, 115, 0.15)',

#
    name='<b>Product Revenue</b>',
    mode='lines+markers',
    yaxis='y2',

    hovertemplate='<b>Product Revenue:</b> $%{y:,.0f}<extra></extra>'
))
fig.update_layout(
    title=dict(
        text='<b>📊 Quantity Sold vs Product Revenue</b>',x=0.5),
    xaxis=dict(title=dict(text='<b>Product</b>',font=dict(size=16))),
    yaxis=dict(
        title='<b>Quantity Ordered</b>',
        showgrid=True,
        gridcolor='lightgray',
        range=[-50, 33000]
    ),
    yaxis2=dict(
        title='<b>Average Price ($)</b>',
        overlaying='y',
        side='right'),
    legend=dict(
        orientation="v",
        x=0.87,
        y=1.2),
    template='plotly_white',
    hovermode='x unified')
fig.show()

### 3️⃣ Revenue Concentration Risk

    Percentage(%) revenue contribution of top 5 products

In [51]:
# Each Product Revenue
top_5_per_revenue = df.groupby('Product')['Total Revenue'].sum().reset_index()

# Calculate Revenue in Percentage(%)
top_5_per_revenue['% Revenue'] = (top_5_per_revenue['Total Revenue'] / (top_5_per_revenue['Total Revenue'].sum())) * 100

# Arrange in Order (Highest to Lowest)
top_5_per_revenue = top_5_per_revenue.sort_values(by='% Revenue',ascending=False).head(5)

percentage_revenue = top_5_per_revenue.copy()

# Write in Proper Readable Format
percentage_revenue['Total Revenue'] = percentage_revenue['Total Revenue'].apply(lambda x: f'$ {x:,.0f}')

# Write in Proper Readable Format
percentage_revenue['% Revenue'] = percentage_revenue['% Revenue'].apply(lambda x: f'{x:,.1f} %')

percentage_revenue

,Product,Total Revenue,% Revenue
13,Macbook Pro Laptop,"$ 7,867,600",23.4 %
18,iPhone,"$ 4,710,300",14.0 %
14,ThinkPad Laptop,"$ 4,037,960",12.0 %
9,Google Phone,"$ 3,253,800",9.7 %
1,27in 4K Gaming Monitor,"$ 2,385,959",7.1 %


    Check if revenue is concentrated

In [26]:
# Top 5 Products
top_5_revenue_concentrated = top_5_per_revenue.sort_values(by='% Revenue',ascending=False).head().reset_index()

# Sum of there Revenue Percentage
top_5_per = round(top_5_revenue_concentrated['% Revenue'].sum(),2)

print(f"Top 5 products generate: {top_5_per} % of Total Revenue.")

Top 5 products generate: 65.86 % of Total Revenue.


In [27]:
fig = go.Figure(go.Pie(
    labels=top_5_revenue_concentrated['Product'],
    values=top_5_revenue_concentrated['% Revenue'],

    textinfo='percent',
    texttemplate='<b> %{value:,.1f}%</b>',
    hole=0.55,
    pull=[0.05, 0.04, 0.03, 0.02, 0.01],

    hoverlabel=dict(font_color='white'),
    textfont=dict(color='white',size=14),
    marker=dict(colors=['#0A2540', '#1E88E5', '#43A0999','#8E24AA'],
    line=dict(color='white', width=2)),

    hovertemplate=
        '<b>Product:</b> %{label}<br>' +
        '<b>Revenue Share:</b> %{value:.1f}%<extra></extra>'
    ))
fig.update_layout(
    title=dict(
        text="<b>Revenue Concentration – Top 5 Products</b>",
        x=0.5,
        font=dict(size=22)),

    annotations = [dict(
        text=f"<b>{top_5_per}%<br>of Total Revenue</b>",
        x=0.5, y=0.5,
        font=dict(size=18),
        showarrow=False)],

    showlegend=True,
    legend=dict(
        orientation="v",
        x=0.77,
        y=1.05))
fig.show()

### 4️⃣ Time-Based Sales Behavior

    Orders by hour

In [48]:
# Orders By Hour
order_by_hours = df.groupby('Hour')['Quantity Ordered'].sum().reset_index()

order_by_hours

,Hour,Quantity Ordered
0,0,4259
1,1,2519
2,2,1349
3,3,900
4,4,897
5,5,1424
6,6,2692
7,7,4365
8,8,6762
9,9,9428


In [29]:
# Arrange in Decending Order
order_by_hours.sort_values(by='Quantity Ordered',ascending=False).head(5)

,Hour,Quantity Ordered
19,19,14470
12,12,14202
11,11,14005
18,18,13802
20,20,13768


In [30]:
# Arrange in Ascending Order
order_by_hours.sort_values(by='Quantity Ordered',ascending=True).head(5)

,Hour,Quantity Ordered
3,3,928
4,4,937
2,2,1398
5,5,1493
1,1,2619


    Revenue by hour

In [49]:
# Revenue By Hour
revenue_by_hours = df.groupby('Hour')['Total Revenue'].sum().reset_index()

revenue_by_hours

,Hour,Total Revenue
0,0,698026.38
1,1,448714.66
2,2,229557.77
3,3,142703.69
4,4,157537.93
5,5,223563.76
6,6,439315.79
7,7,729061.89
8,8,1168940.67
9,9,1595857.04


In [32]:
# Arrange in Order (Highest to Lowest)
revenue_by_hours.sort_values(by='Total Revenue',ascending=False).head(5)

,Hour,Total Revenue
19,19,2412938.54
12,12,2316821.34
11,11,2300610.24
20,20,2281716.24
18,18,2219348.30


In [33]:
# Arrange in Order (Lowest to Highest)
revenue_by_hours.sort_values(by='Total Revenue',ascending=True).head(5)

,Hour,Total Revenue
3,3,145757.89
4,4,162661.01
5,5,230679.82
2,2,234851.44
6,6,448113.00


    Identify peak hours

In [34]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Order By Hours</b>',
                    '<b>Revenue By Hours</b>'))
fig.add_trace(go.Scatter(
    x=order_by_hours['Hour'],
    y=order_by_hours['Quantity Ordered'],

    mode='lines+markers',
    name='<b>Orders</b>',

    line=dict(
        color='green',
        width=4,
        shape='spline'),
    marker=dict(
        size=8,
        color='black',
        line=dict(width=3, color='white')),
    fill='tozeroy',
    fillcolor='rgba(60, 234, 115, 0.15)',

    hovertemplate='<b>Hour:</b> %{x}<br><b>Orders:</b> %{y:,.0f}<extra></extra>'),
    row=1, col=1)

fig.add_trace(go.Scatter(
    x=revenue_by_hours['Hour'],
    y=revenue_by_hours['Total Revenue'],

    mode='lines+markers',
    name='<b>Revenue</b>',

    line=dict(
        color='#1f77b4',
        width=4,
        shape='spline'),
    marker=dict(
        size=8,
        color='black',
        line=dict(width=3, color='white')),
    fill='tozeroy',
    fillcolor='rgba(30, 136, 229, 0.15)',

    hovertemplate='<b>Hour:</b> %{x}<br><b>Revenue:</b> $%{y:,.0f}<extra></extra>'),
    row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Quantity Ordered & Revenue Trend By Hours</b>',x=0.5,font=dict(size=22)),
    showlegend=True,
    template='simple_white',
    legend=dict(
        x=0.9,
        y=1.3,
        bgcolor='rgba(255,255,255,0.8)'),
    hovermode='x unified',
    height=400)

fig.update_xaxes(title_text="<b>Hours</b>", row=1, col=1, showgrid=True)
fig.update_xaxes(title_text="<b>Hours</b>", row=1, col=2, showgrid=True)
fig.update_yaxes(title_text="<b>Quantity Ordered</b>", row=1, col=1, showgrid=True)
fig.update_yaxes(title_text="<b>Revenue</b>", row=1, col=2, showgrid=True)
fig.show()

### 5️⃣ City Performance

    Revenue by city

In [35]:
# Revenue By Cities
city_revenue = df.groupby('City')['Total Revenue'].sum().reset_index()

# Arrange in Order (Highest to Lowest)
city_revenue.sort_values(by='Total Revenue',ascending=False)

,City,Total Revenue
8,San Francisco (CA),8262203.91
4,Los Angeles (CA),5452570.80
5,New York City (NY),4664317.43
2,Boston (MA),3661642.01
0,Atlanta (GA),2795498.58
3,Dallas (TX),2767975.40
9,Seattle (WA),2747755.48
7,Portland (OR),1870732.34
1,Austin (TX),1819581.75
6,Portland (ME),449758.27


    Order count by city

In [36]:
# Quantity Ordered By Cities
city_order = df.groupby('City')['Order ID'].count().reset_index()

# Rename Column
city_order.rename(columns={'Order ID': 'Quantity Ordered'},inplace=True)

# Arrange in Order (Highest to Lowest)
city_order.sort_values(by='Quantity Ordered',ascending=False)

,City,Quantity Ordered
8,San Francisco (CA),44732
4,Los Angeles (CA),29605
5,New York City (NY),24876
2,Boston (MA),19934
0,Atlanta (GA),14881
3,Dallas (TX),14820
9,Seattle (WA),14732
7,Portland (OR),10010
1,Austin (TX),9905
6,Portland (ME),2455


In [37]:
city_analysis = city_revenue.merge(city_order, on='City')
city_analysis = city_analysis.sort_values(by='Total Revenue',ascending=False)
city_analysis['Revenue_M'] = city_analysis['Total Revenue'] / 1_000_000

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(
    x=city_analysis['City'],
    y=city_analysis['Revenue_M'],

    name='<b>Total Revenue</b>',
    text=city_analysis['Revenue_M'].apply(lambda x: f"<b>${x:.1f}M</b>"),
    textposition='outside',

    marker=dict(
        color=city_analysis['Revenue_M'],
        colorscale='thermal',
        line=dict(color='white', width=1.5)),

    yaxis='y1',
    hovertemplate='<b>%{y:.2f}M</b>'))

fig.add_trace(go.Scatter(
    x=city_analysis['City'],
    y=city_analysis['Quantity Ordered'],

    name='<b>Total Orders</b>',
    text=city_analysis['Quantity Ordered'].apply(lambda x: f"<b>{x:,}</b>"),
    textposition='top center',

    line=dict(
        color='#1E88E5',
        width=3,
        shape='spline'),
    marker=dict(
        size=8,
        color='#0A2540',
        line=dict(color='white', width=2)),

    yaxis='y2',
    hovertemplate='<b>Total Orders: %{y:,}</b><extra></extra>'))
    
fig.update_layout(
    title=dict(
        text='<b>City Performance: Revenue vs Orders</b>',x=0.5,font=dict(size=24)),
    xaxis=dict(
        title=dict(text='<b>Cities</b>', font=dict(size=20)),showgrid=False,tickfont=dict(size=12)),

    yaxis=dict(
      title=dict(
            text='<b>Total Revenue (Millions $)</b>',
            font=dict(size=16)),
      showgrid=True,gridcolor='rgba(0,0,0,0.11)',ticksuffix='M',tickfont=dict(size=14)),

    yaxis2=dict(
        title=dict(text='<b>Total Orders</b>',font=dict(size=16)),overlaying='y',side='right',tickfont=dict(size=14)),

    template='simple_white',
    hovermode='x unified',
    bargap=0.22,
    legend=dict(
        x=0.78,
        y=1.15,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1),
    margin=dict(t=90, l=70, r=30, b=60))
fig.show()

    Average order value per city

In [38]:
# Each City Orders and Revenue
aov_city = df.groupby('City').agg(total_revenue = ('Total Revenue', 'sum'),unique_order = ('Order ID', 'nunique')).reset_index()

# Add Columns, Average Number of Orders Each City Ordered
aov_city['Average_Order_Value'] = (aov_city['total_revenue'] / aov_city['unique_order'])

# Give Columns Proper Name
aov_city = aov_city.rename(columns={'total_revenue': 'Total Revenue','unique_order': 'Total Orders'})

# Arrange in Order (Highest to Lowest)
aov_city = aov_city.sort_values(by='Average_Order_Value',ascending=False)

In [39]:
# Write in Proper Readable Format
aov_city['Total Revenue'] = aov_city['Total Revenue'].apply(lambda x: f'{x:,.0f}')
aov_city['Total Orders'] = aov_city['Total Orders'].apply(lambda x: f'{x:,.0f}')
aov_city['Average_Order_Value'] = aov_city['Average_Order_Value'].apply(lambda x: f'{x:,.0f}')

aov_city

,City,Total Revenue,Total Orders,Average_Order_Value
0,Atlanta (GA),"2,795,499","14,253",196
5,New York City (NY),"4,664,317","23,848",196
9,Seattle (WA),"2,747,755","14,119",195
7,Portland (OR),"1,870,732","9,617",195
3,Dallas (TX),"2,767,975","14,240",194
8,San Francisco (CA),"8,262,204","42,898",193
2,Boston (MA),"3,661,642","19,092",192
1,Austin (TX),"1,819,582","9,509",191
4,Los Angeles (CA),"5,452,571","28,498",191
6,Portland (ME),"449,758","2,363",190


### 6️⃣ Market Basket Analysis

    Find orders with multiple products (same Order ID)

In [40]:
# Select Multiple Products
df['Sold Together'] = df.groupby('Order ID')['Product'].transform(lambda x: ', '.join(x))

# Drop Duplicate Order ID's
df = df.drop_duplicates(subset=['Order ID'], ignore_index=True)

In [41]:
# Count Number of Products Sold (single or Together Both)
multi_products = df['Sold Together'].value_counts().reset_index()

# Display Specfic Column Details
multi_products.columns = ['Products Sold Together', 'Count']

# Count Number of Products Sold (Together)
together_products = multi_products[multi_products['Products Sold Together'].str.contains(', ')]

together_products.head(10)

,Products Sold Together,Count
17,"iPhone, Lightning Charging Cable",882
18,"Google Phone, USB-C Charging Cable",856
21,"iPhone, Wired Headphones",361
22,"Vareebadd Phone, USB-C Charging Cable",312
23,"Google Phone, Wired Headphones",303
24,"iPhone, Apple Airpods Headphones",286
25,"Google Phone, Bose SoundSport Headphones",161
26,"Vareebadd Phone, Wired Headphones",104
27,"Google Phone, USB-C Charging Cable, Wired Head...",77
28,"Vareebadd Phone, Bose SoundSport Headphones",60


    Identify frequently bought together products

In [42]:
top_combo = together_products.head(10)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=top_combo['Products Sold Together'][::-1],
    x=top_combo['Count'][::-1],

    orientation='h',
    text=top_combo['Count'][::-1],
    texttemplate='<b>%{text:,}</b>',
    textposition='outside',

    marker=dict(
        color=top_combo['Count'][::-1],
        colorscale='ylorrd',
        line=dict(color='white', width=1.5)),

    hovertemplate=
    '<b>Products:</b> %{y}<br>' +
    '<b>Orders Together:</b> %{x:,}<extra></extra>'
    ))
fig.update_layout(
    title=dict(
        text='<b>Top Products Frequently Purchased Together</b>',
        x=0.5,
        font=dict(size=24)),

    xaxis=dict(
        title=dict(text='<b>Number of Orders</b>', font=dict(size=18)),showgrid=True,gridcolor='rgba(0,0,0,0.20)'),

    yaxis=dict(
        title=dict(text='<b>Number of Orders</b>', font=dict(size=18))),

    template='simple_white',
    bargap=0.35,
    margin=dict(
        l=400,
        r=40,
        t=80,
        b=50))
fig.show()

### 7️⃣ Time-Based Product Demand

Which products sell more:

    By Hour

In [43]:
# Product sales by hour
product_sales_per_hour = df.groupby(['Hour', 'Product']).agg(Quantity_Sold=('Quantity Ordered', 'sum'),Revenue=('Total Revenue', 'sum')).reset_index()

# Top 5 best-selling products
top_products = df.groupby('Product')['Quantity Ordered'].sum().sort_values(ascending=False).head(5).index

# Filter top products only
top_hourly = product_sales_per_hour[product_sales_per_hour['Product'].isin(top_products)]

top_hourly

,Hour,Product,Quantity_Sold,Revenue
4,0,AA Batteries (4-pack),521,2000.64
5,0,AAA Batteries (4-pack),702,2098.98
12,0,Lightning Charging Cable,410,6129.50
15,0,USB-C Charging Cable,476,5688.20
17,0,Wired Headphones,401,4807.99
...,...,...,...,...
441,23,AA Batteries (4-pack),937,3598.08
442,23,AAA Batteries (4-pack),995,2975.05
449,23,Lightning Charging Cable,716,10704.20
452,23,USB-C Charging Cable,710,8484.50


In [44]:
fig = go.Figure()

for product in top_products:

    temp = top_hourly[top_hourly['Product'] == product]

    fig.add_trace(go.Scatter(
        x=temp['Hour'],
        y=temp['Quantity_Sold'],

        mode='lines+markers',
        name=product,
        line=dict(width=4, shape='spline'),
        marker=dict(
            size=8,
            line=dict(color='white', width=2)),

        hovertemplate=
        '<b>Product:</b> ' + product + '<br>' +
        '<b>Hour:</b> %{x}:00<br>' +
        '<b>Quantity Sold:</b> %{y:,}<extra></extra>'))

fig.update_layout(
    title=dict(
        text='<b>Top Products Sales Trend by Hour</b>',
        x=0.5,
        font=dict(size=24)),

    xaxis=dict(
        title=dict(text='<b>Hours</b>', font=dict(size=18)),
        tickmode='linear',
        dtick=1,
        showgrid=False),

    yaxis=dict(
        title=dict(text='<b>Quantity Sold</b>',font=dict(size=16)),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.15)'),

    legend=dict(
        title='<b>Products</b>',
        x=0.92,
        y=1.1,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1),

    template='simple_white',
    hovermode='x unified',
    margin=dict(t=80, l=90, r=150, b=70))
fig.show()

    By Month

In [45]:
# Product sales by Month
product_sales_per_month = df.groupby(['Month', 'Product']).agg(Quantity_Sold=('Quantity Ordered', 'sum'),Revenue=('Total Revenue', 'sum')).reset_index()

# Top 5 best-selling products
top_products = df.groupby('Product')['Quantity Ordered'].sum().sort_values(ascending=False).head(5).index

# Filter top products only
top_monthly = product_sales_per_month[product_sales_per_month['Product'].isin(top_products)]

top_monthly.head()

,Month,Product,Quantity_Sold,Revenue
4,1,AA Batteries (4-pack),1394,5352.96
5,1,AAA Batteries (4-pack),1583,4733.17
12,1,Lightning Charging Cable,1074,16056.30
15,1,USB-C Charging Cable,1174,14029.30
17,1,Wired Headphones,982,11774.18


In [46]:
fig = go.Figure()

for product in top_products:

    temp = top_monthly[top_monthly['Product'] == product]

    fig.add_trace(go.Scatter(
        x=temp['Month'],
        y=temp['Quantity_Sold'],

        mode='lines+markers',
        name=product,
        line=dict(width=4, shape='spline'),
        marker=dict(
            size=8,
            line=dict(color='white', width=2)),

        hovertemplate=
        '<b>Product:</b> ' + product + '<br>' +
        '<b>Month:</b> %{x}:00<br>' +
        '<b>Quantity Sold:</b> %{y:,}<extra></extra>'))

fig.update_layout(
    title=dict(
        text='<b>Top Products Sales Trend by Months</b>',
        x=0.5,
        font=dict(size=24)),

    xaxis=dict(
        title=dict(text='<b>Months</b>', font=dict(size=18)),
        tickmode='linear',
        dtick=1,
        showgrid=False),

    yaxis=dict(
        title=dict(text='<b>Quantity Sold</b>',font=dict(size=16)),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.15)'),

    legend=dict(
        title='<b>Products</b>',
        x=0.97,
        y=1.11,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1),

    template='simple_white',
    hovermode='x unified',
    margin=dict(t=80, l=90, r=150, b=70))
fig.show()

## Business Recommendations(Suggestions)

### 1️⃣ Reduce Revenue Concentration Risk

The top 5 products contribute approximately 65.86% of total revenue, which creates a high business dependency risk. If demand for one major product drops, overall company revenue could decline significantly.

    Recommendation
- Expand marketing campaigns for low-performing products such as cables, headphones, and batteries.
- Create product bundles with high-demand devices (e.g., MacBook + accessories).
- Introduce cross-selling strategies during checkout.

### 2️⃣ Increase Sales During Low-Revenue Months

Sales performance declines noticeably during May, June, August, and September, while October and December generate peak growth.

    Recommendation
- Launch seasonal campaigns and limited-time discounts during slow months.
- Run mid-year promotions, back-to-school campaigns, and bundle offers.
- Increase digital advertising spend before weak-performing months.

### 3️⃣ Optimize Sales Around Peak Hours

The highest order volume and revenue occur between 11 AM – 8 PM, especially around 7 PM.

    Recommendation
- Schedule paid advertisements and email campaigns during peak conversion hours.
- Ensure inventory and customer support availability during high-demand periods.
- Use flash sales during peak traffic windows.

### 4️⃣ Improve Performance in Low-Revenue Cities

Cities like Portland (ME) and Austin (TX) generate significantly lower revenue compared to San Francisco and Los Angeles.

    Recommendation
- Conduct localized marketing campaigns in underperforming cities.
- Offer city-specific promotions and free shipping incentives.
- Analyze regional customer preferences and product demand.

### 5️⃣ Leverage Market Basket Opportunities

Customers frequently purchase products together, such as:

1. iPhone + Lightning Charging Cable
2. Google Phone + USB-C Charging Cable
3. iPhone + Wired Headphones

        Recommendation

- Introduce “Frequently Bought Together” recommendations on the website.
- Offer bundle discounts for commonly paired products.
- Use AI-based recommendation systems for personalized upselling.

### 6️⃣ Improve Inventory Planning

Products such as batteries and charging cables sell in very high quantities but generate relatively lower revenue individually.

    Recommendation
- Maintain higher inventory levels for fast-moving accessories.
- Use demand forecasting models for stock optimization.
- Prevent stockouts during peak sales hours and months.

### 7️⃣ Focus on High-Value Products

Premium products like the MacBook Pro Laptop, iPhone, and ThinkPad Laptop generate the highest revenue despite lower order quantities.

    Recommendation
- Prioritize premium product advertising campaigns.
- Offer financing or installment payment options.
- Create loyalty rewards for high-value customers.